# Практика · Перенос навчання

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ Зошит навчає **понад пʼятдесят** мереж: дві великі джерельні по 1500 прикладів
> і решту маленьких, від 12 до 240 прикладів. Заміряно на чотирьох ядрах без
> відеокарти: **півтори хвилини** на вільній машині й до чотирьох на завантаженій.
> Сам рахунок іде в один потік — так швидше, див. нижче.

У лекції ми стверджували чотири речі. Тут ти отримаєш кожну з них числом.

1. **Навчимо джерельну модель** на трьох фігурах — це наші власні «передтреновані ваги».
2. **Збережемо й завантажимо** їх через `state_dict` — щоб побачити сам механізм переносу.
3. **Замір 1 — розворот:** перенос виграє на малій цілі й програє на достатній.
4. **Замір 2 — донавчання проти замороженого тіла:** вища стеля, але більший розкид.
5. **Замір 3 — що переноситься:** заморозимо різну кількість блоків.
6. **Замір 4 — коли перенос шкодить:** візьмемо джерело, несхоже на ціль.
7. **Перші фільтри** джерела поруч із фільтрами моделі, навченої з нуля.
8. **Не та нормалізація** — і скільки точності вона коштує.

**Мережа не потрібна:** датасет ми малюємо самі, формулами. Передтренованих ваг
із ImageNet тут немає свідомо — вони важать 44,7 МБ і тягнуться з інтернету.

In [ ]:
import time
import copy
import numpy as np
import torch
import torch.nn as nn

# зерна фіксуємо на самому початку: без них числа нижче не збіжаться з лекцією
torch.manual_seed(0)
rng = np.random.default_rng(42)

# один потік, а не чотири. Мережа тут крихітна, і на батчі з восьми зображень
# чотири потоки більше часу домовляються між собою, ніж рахують: заміряно
# 34,5 мс на крок в один потік проти 1226 мс у чотири на завантаженій машині.
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## 1 · Дві задачі з одного світу

Переносити можна лише тоді, коли є **звідки** й **куди**. Тому задач у нас дві.

- **Джерельна задача** — три фігури: коло, квадрат, ромб. Даних багато: 1500 прикладів.
- **Цільова задача** — три **інші** фігури: кільце, хрест, трикутник. Даних мало,
  і саме це ми будемо міняти: від 12 до 240 прикладів.

Класи не перетинаються. Модель, навчена на колі й квадраті, ніколи не бачила хреста —
переносяться не відповіді, а вміння дивитися.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]
SOURCE_KINDS = [0, 1, 2]        # коло, квадрат, ромб
TARGET_KINDS = [3, 4, 5]        # кільце, хрест, трикутник


def draw_shape(kind, rng, size=28, jitter=5, noise=0.60):
    """Малює одну фігуру заданого класу як масив 28×28 зі значеннями 0..1.

    Шум великий (σ=0.6) і зсув теж (±5 пікселів) — і це навмисно. На чистих
    фігурах задача розвʼязується з дванадцяти прикладів, і жодної різниці між
    способами навчання не видно: усі дають одиницю.
    """
    image = np.zeros((size, size), dtype=np.float32)
    # центр зсуваємо на кілька пікселів, щоб мережа не завчила одне положення
    center_y = size / 2 + rng.integers(-jitter, jitter + 1)
    center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    radius = rng.integers(7, 10)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_dataset(count, kinds, rng):
    """Повертає (count, 1, 28, 28) і (count,). Мітки — 0..len(kinds)-1, порівну."""
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        slot = i % len(kinds)                  # рівно по третині кожного класу
        images[i, 0] = draw_shape(kinds[slot], rng)
        labels[i] = slot
    return torch.from_numpy(images), torch.from_numpy(labels)


source_x, source_y = make_dataset(1500, SOURCE_KINDS, rng)
target_pool_x, target_pool_y = make_dataset(240, TARGET_KINDS, rng)
test_x, test_y = make_dataset(300, TARGET_KINDS, rng)

print("джерело      :", tuple(source_x.shape), "мітки", sorted(set(source_y.tolist())))
print("цільовий пул :", tuple(target_pool_x.shape))
print("цільовий тест:", tuple(test_x.shape))

Подивимось на обидві трійки очима. Верхній рядок — джерело, нижній — ціль.
Фігури різні, але світ один: біла фігура на чорному тлі, трохи зсунута, з шумом.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(2, 3, figsize=(6, 4.2))
for slot in range(3):
    axes[0, slot].imshow(source_x[slot, 0], cmap="gray")
    axes[0, slot].set_title("джерело: " + SHAPE_NAMES[SOURCE_KINDS[slot]], fontsize=9)
    axes[1, slot].imshow(target_pool_x[slot, 0], cmap="gray")
    axes[1, slot].set_title("ціль: " + SHAPE_NAMES[TARGET_KINDS[slot]], fontsize=9)
    axes[0, slot].axis("off")
    axes[1, slot].axis("off")
plt.tight_layout()
plt.show()
print("три класи в джерелі й три ІНШІ класи в цілі — спільних відповідей немає")

## 2 · Мережа: тіло і голова

Мережа з теми 09, розділена на дві частини **явно**, окремими полями:

- **тіло** (`body`) — три блоки `Conv → ReLU → Pool`. Воно перетворює зображення
  28×28 на 288 чисел: 32 канали по 3×3. Це і є представлення.
- **голова** (`head`) — `Flatten` і два повнозвʼязні шари. Вона ухвалює рішення.

Перенос — це буквально «лишити `body`, замінити `head`». Щоб так можна було
зробити, розділення треба закласти в код заздалегідь.

In [ ]:
def conv_block(in_channels, out_channels):
    """Один типовий блок: Conv → ReLU → Pool (порядок з теми 09)."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


def make_head(n_classes=3):
    """Нова голова: 288 чисел від тіла → 64 → кількість класів."""
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 3 * 3, 64),
        nn.ReLU(),
        nn.Linear(64, n_classes),
    )


class ShapeNet(nn.Module):
    """Тіло й голова — окремі поля, щоб голову можна було замінити одним рядком."""

    def __init__(self, n_classes=3):
        super().__init__()
        self.body = nn.Sequential(
            conv_block(1, 8),      # 1×28×28  →  8×14×14
            conv_block(8, 16),     # 8×14×14  → 16×7×7
            conv_block(16, 32),    # 16×7×7   → 32×3×3
        )
        self.head = make_head(n_classes)

    def forward(self, x):
        return self.head(self.body(x))


torch.manual_seed(0)
probe = ShapeNet(3)
body_params = sum(p.numel() for p in probe.body.parameters())
head_params = sum(p.numel() for p in probe.head.parameters())

# ручний рахунок голови: 288×64 ваг + 64 зсуви + 64×3 ваг + 3 зсуви
head_by_hand = 288 * 64 + 64 + 64 * 3 + 3

print("параметрів у тілі  :", body_params)
print("параметрів у голові:", head_params, "  рукою:", head_by_hand)
assert head_params == head_by_hand, "ручний рахунок параметрів голови розійшовся!"
print("✅ збігається")
print()
print("частка голови в мережі: %.1f %%" % (100 * head_params / (body_params + head_params)))

## 3 · Навчаємо джерельну модель

Це найдовша клітинка зошита — **близько 25 секунд**. Саме вона й робить те, що
в реальному житті зробив хтось інший на ImageNet: перетворює купу даних на ваги.

Функція `train` нижче отримує окремий аргумент `params` — список параметрів, які
вільно вчитися. Це знадобиться, коли ми заморозимо тіло.

In [ ]:
def train(model, x, y, epochs, learning_rate, batch=16, seed=0, params=None):
    """Навчає модель. params=None означає «вчити все, що є в моделі»."""
    torch.manual_seed(seed)                      # порядок батчів теж має бути відтворюваним
    optimizer = torch.optim.Adam(
        params if params is not None else model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        order = torch.randperm(len(x))
        for start in range(0, len(x), batch):
            batch_index = order[start:start + batch]
            optimizer.zero_grad()
            loss = loss_function(model(x[batch_index]), y[batch_index])
            loss.backward()
            optimizer.step()
    return model


@torch.no_grad()
def accuracy(model, x, y):
    """Частка правильних відповідей."""
    model.eval()
    return (model(x).argmax(1) == y).float().mean().item()


torch.manual_seed(0)
source_model = ShapeNet(3)

started = time.perf_counter()
train(source_model, source_x, source_y, epochs=10, learning_rate=1e-3, batch=32, seed=0)
source_seconds = time.perf_counter() - started

print("навчання джерельної моделі: %.1f с" % source_seconds)
print("точність на джерельній задачі: %.3f" % accuracy(source_model, source_x, source_y))

## 4 · `state_dict`: як ваги переїжджають між моделями

Ваги моделі в PyTorch — це звичайний словник: імʼя шару → тензор чисел.
Він зветься `state_dict`, і саме його зберігають у файл `.pt` або `.pth`.
Коли ти пишеш `resnet18(weights="IMAGENET1K_V1")`, бібліотека завантажує
з інтернету рівно такий словник і кладе його в порожню модель.

Зробимо те саме руками — щоб механізм перестав бути магією.

In [ ]:
weights = source_model.state_dict()

print("ключів у state_dict:", len(weights))
print()
print("%-24s %-20s %8s" % ("імʼя", "форма", "чисел"))
print("-" * 54)
for name, tensor in weights.items():
    print("%-24s %-20s %8d" % (name, tuple(tensor.shape), tensor.numel()))

Тепер збережемо словник у файл і завантажимо його в **іншу**, щойно створену модель.
Перевірка проста: ваги в обох мають збігтися до останнього біта, а відповіді на
тих самих зображеннях — теж.

In [ ]:
torch.save(source_model.state_dict(), "source_weights.pt")

import os
print("файл на диску: %.1f КБ" % (os.path.getsize("source_weights.pt") / 1024))

# нова модель із випадковими вагами — поки вона нічого не вміє
torch.manual_seed(123)
restored = ShapeNet(3)
print("до завантаження, точність на джерелі: %.3f" % accuracy(restored, source_x, source_y))

restored.load_state_dict(torch.load("source_weights.pt", weights_only=True))
print("після завантаження, точність на джерелі: %.3f" % accuracy(restored, source_x, source_y))

# звірка «наше = бібліотечне»: ваги мають збігтися побітово
same = all(torch.equal(a, b) for a, b in
           zip(source_model.state_dict().values(), restored.state_dict().values()))
assert same, "ваги після завантаження розійшлися!"
print("✅ усі", len(weights), "тензорів збіглися побітово")

# зберігаємо копію в памʼяті: далі кожен дослід починається з тих самих ваг
source_state = copy.deepcopy(source_model.state_dict())

os.remove("source_weights.pt")            # файл своє показав, теку лишаємо чистою
print("тимчасовий файл ваг прибрано")

## 5 · Три способи переносу

Тепер три функції, і всі три відрізняються буквально трьома рядками.

| Спосіб | Що з тілом | Що вчиться |
|---|---|---|
| З нуля | нове випадкове | усе |
| Заморожене тіло | ваги джерела, `requires_grad = False` | лише голова |
| Донавчання | ваги джерела, градієнти дозволені | усе |

Голова в другому й третьому випадку **завжди нова**: стара вміє відповідати
«коло / квадрат / ромб», а нам треба «кільце / хрест / трикутник».

In [ ]:
def build_from_scratch():
    """Порожня мережа: жодного знання ззовні."""
    return ShapeNet(3)


def build_transferred(state, freeze_body=True, frozen_blocks=3):
    """Мережа з чужим тілом і новою головою.

    frozen_blocks — скільки ПЕРШИХ блоків тіла заморозити (0..3).
    freeze_body=False означає донавчання: тіло чуже, але вільне.
    """
    model = ShapeNet(3)
    model.load_state_dict(state)          # тіло й голова джерела
    model.head = make_head(3)             # стару голову викидаємо, ставимо чисту
    if freeze_body:
        for block_index in range(frozen_blocks):
            for parameter in model.body[block_index].parameters():
                parameter.requires_grad = False
    return model


def trainable(model):
    """Список параметрів, яким дозволено змінюватись."""
    return [p for p in model.parameters() if p.requires_grad]


def run(model, n_examples, epochs, learning_rate, seed=1):
    """Навчає модель на перших n прикладах цілі й повертає точність на тесті."""
    x, y = target_pool_x[:n_examples], target_pool_y[:n_examples]
    train(model, x, y, epochs, learning_rate, batch=8, seed=seed, params=trainable(model))
    return accuracy(model, test_x, test_y)


# скільки чисел навчається в кожному режимі й скільки коштує одне навчання
MODES = [
    ("з нуля", lambda: build_from_scratch(), 3e-3),
    ("заморожене тіло", lambda: build_transferred(source_state), 3e-3),
    ("донавчання", lambda: build_transferred(source_state, freeze_body=False), 5e-4),
]

mode_cost = {}
print("%-18s %12s %10s %10s" % ("спосіб", "навчуваних", "секунд", "точність"))
print("-" * 54)
for name, make_model, learning_rate in MODES:
    torch.manual_seed(1)
    model = make_model()
    free = sum(p.numel() for p in trainable(model))
    started = time.perf_counter()
    score = run(model, 24, epochs=20, learning_rate=learning_rate)
    seconds = time.perf_counter() - started
    mode_cost[name] = (free, seconds, score)
    print("%-18s %12d %10.2f %10.3f" % (name, free, seconds, score))

## 6 · Замір 1: розворот на розмірі цільових даних

Головний дослід теми. Беремо 12, 24, 60, 120 і 240 цільових прикладів і в кожній
точці навчаємо три моделі: з нуля, із замороженим тілом, із донавчанням.
Тест той самий — 300 зображень, яких не бачила жодна з них.

Пʼятнадцять навчань, кожне від секунди до десяти.

In [ ]:
SIZES = (12, 24, 60, 120, 240)
measure_1 = {}

started = time.perf_counter()
for n in SIZES:
    torch.manual_seed(1)
    from_scratch = run(build_from_scratch(), n, epochs=20, learning_rate=3e-3)
    torch.manual_seed(1)
    frozen = run(build_transferred(source_state), n, epochs=20, learning_rate=3e-3)
    torch.manual_seed(1)
    fine_tuned = run(build_transferred(source_state, freeze_body=False), n,
                     epochs=20, learning_rate=5e-4)
    measure_1[n] = (from_scratch, frozen, fine_tuned)

print("%10s %10s %14s %12s" % ("прикладів", "з нуля", "заморожене", "донавчання"))
print("-" * 50)
for n in SIZES:
    a, b, c = measure_1[n]
    lead = "  ← перенос виграє" if b > a else "  ← перенос ПРОГРАЄ"
    print("%10d %10.3f %14.3f %12.3f%s" % (n, a, b, c, lead))
print()
print("рівень випадкового вгадування на трьох класах: 0.333")
print("усього на замір: %.1f с" % (time.perf_counter() - started))

Тепер знайдемо **точку розвороту** — найменший розмір вибірки, при якому навчання
з нуля вже не гірше за перенос. Це і є практична відповідь на питання
«чи брати мені передтреновану модель».

In [ ]:
crossover = None
for n in SIZES:
    from_scratch, frozen, _ = measure_1[n]
    if from_scratch >= frozen:
        crossover = n
        break

if crossover is None:
    print("на всіх перевірених розмірах перенос попереду — розворот далі за 120 прикладів")
else:
    print("розворот стався на", crossover, "прикладах")
    before = SIZES[SIZES.index(crossover) - 1]
    print("на %d прикладах: з нуля %.3f проти переносу %.3f" % (before, *measure_1[before][:2]))
    print("на %d прикладах: з нуля %.3f проти переносу %.3f" % (crossover, *measure_1[crossover][:2]))
    print()
    print("висновок: перенос — це ліки від нестачі даних, а не безумовне покращення")

## 7 · Замір 2: донавчання проти замороженого тіла — чесно

У заміру 1 донавчання скрізь гірше. Спокуслива відповідь: «заморожене тіло
краще». Неправильна: донавчання ми запускали з `lr=5e-4`, і це для нього
поганий режим.

Перевіримо на 24 прикладах, як обидва способи реагують на налаштування.

In [ ]:
measure_2 = []

# та сама сітка налаштувань для обох способів — інакше порівняння нечесне
for mode in ("заморожене тіло", "донавчання"):
    for learning_rate in (5e-4, 1e-3, 3e-3):
        for epochs in (20, 40):
            torch.manual_seed(1)
            model = build_transferred(source_state,
                                      freeze_body=(mode == "заморожене тіло"))
            measure_2.append((mode, learning_rate, epochs,
                              run(model, 24, epochs, learning_rate)))

print("%-18s %8s %7s %10s" % ("режим", "lr", "епох", "точність"))
print("-" * 46)
for mode, learning_rate, epochs, score in measure_2:
    print("%-18s %8.0e %7d %10.3f" % (mode, learning_rate, epochs, score))

frozen_scores = [s for m, _, _, s in measure_2 if m == "заморожене тіло"]
tuned_scores = [s for m, _, _, s in measure_2 if m == "донавчання"]
print()
print("заморожене тіло: від %.3f до %.3f, розкид %.3f"
      % (min(frozen_scores), max(frozen_scores), max(frozen_scores) - min(frozen_scores)))
print("донавчання     : від %.3f до %.3f, розкид %.3f"
      % (min(tuned_scores), max(tuned_scores), max(tuned_scores) - min(tuned_scores)))

До заміру 2 варто додати ще один режим, який радять найчастіше: **спершу голова,
потім усе тіло з дуже малим кроком**. Ідея в тому, щоб випадкова голова не встигла
зіпсувати чуже тіло великими градієнтами на першому ж кроці.

In [ ]:
torch.manual_seed(1)
two_stage = build_transferred(source_state, freeze_body=True)
# етап 1: тіло заморожене, голова швидко наздоганяє
train(two_stage, target_pool_x[:24], target_pool_y[:24], 20, 3e-3, batch=8, seed=1,
      params=trainable(two_stage))
# етап 2: розморожуємо все й ідемо дуже дрібним кроком
for parameter in two_stage.parameters():
    parameter.requires_grad = True
train(two_stage, target_pool_x[:24], target_pool_y[:24], 20, 2e-4, batch=8, seed=101,
      params=trainable(two_stage))

two_stage_score = accuracy(two_stage, test_x, test_y)
print("голова, потім усе тіло (lr=2e-4): %.3f" % two_stage_score)
print("найкраще заморожене тіло        : %.3f" % max(frozen_scores))
print("найкраще донавчання             : %.3f" % max(tuned_scores))

## 8 · Замір 3: що саме переноситься

Тіло має три блоки. Заморозимо різну їх кількість — від нуля (усе вільне)
до трьох (усе чуже й нерухоме) — і подивимось, де знання джерела допомагає,
а де заважає.

Літературне очікування: **ранні шари універсальні, пізні специфічні**. Тобто
перший блок можна брати чужим сміливо, а третій — уже під питанням.

In [ ]:
FREEZE_LABELS = {0: "нічого", 1: "перший блок", 2: "перші два", 3: "усе тіло"}
measure_3 = {}

for n in (24, 120):
    for frozen_blocks in (0, 1, 2, 3):
        torch.manual_seed(1)
        model = build_transferred(source_state, freeze_body=True, frozen_blocks=frozen_blocks)
        free = sum(p.numel() for p in trainable(model))
        measure_3[(n, frozen_blocks)] = (run(model, n, epochs=20, learning_rate=3e-3), free)

for n in (24, 120):
    print("цільових прикладів:", n)
    print("%16s %10s %20s" % ("заморожено", "точність", "вільних параметрів"))
    print("-" * 50)
    for frozen_blocks in (0, 1, 2, 3):
        score, free = measure_3[(n, frozen_blocks)]
        print("%16s %10.3f %20d" % (FREEZE_LABELS[frozen_blocks], score, free))
    best = max((0, 1, 2, 3), key=lambda k: measure_3[(n, k)][0])
    print("найкраще: заморозити %s (%.3f)" % (FREEZE_LABELS[best], measure_3[(n, best)][0]))
    print()

## 9 · Замір 4: коли перенос шкодить

Досі джерело було близьке до цілі: ті самі 28×28, біла фігура на чорному тлі.
Тепер візьмемо джерело, максимально **несхоже**: суцільні текстури — горизонтальні
смуги, вертикальні смуги й рівномірний шум. Тут немає ні предмета, ні тла, ні
контуру. Це наш аналог того, як ImageNet виглядає для рентгенівського знімка.

In [ ]:
def draw_texture(kind, rng, size=28):
    """Текстура на весь кадр: смуги по горизонталі, по вертикалі або чистий шум."""
    yy, xx = np.mgrid[0:size, 0:size]
    phase = rng.uniform(0, 2 * np.pi)
    period = rng.integers(3, 7)
    if kind == 0:
        image = 0.5 + 0.5 * np.sin(2 * np.pi * yy / period + phase)
    elif kind == 1:
        image = 0.5 + 0.5 * np.sin(2 * np.pi * xx / period + phase)
    else:
        image = rng.uniform(0, 1, (size, size))
    image = image.astype(np.float32) + rng.normal(0, 0.05, (size, size)).astype(np.float32)
    return np.clip(image, 0, 1)


def make_texture_dataset(count, rng):
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        images[i, 0] = draw_texture(i % 3, rng)
        labels[i] = i % 3
    return torch.from_numpy(images), torch.from_numpy(labels)


far_x, far_y = make_texture_dataset(1500, rng)

figure, axes = plt.subplots(1, 3, figsize=(6, 2.2))
for kind in range(3):
    axes[kind].imshow(far_x[kind, 0], cmap="gray")
    axes[kind].set_title(["горизонталь", "вертикаль", "шум"][kind], fontsize=9)
    axes[kind].axis("off")
plt.tight_layout()
plt.show()
print("далеке джерело: три класи текстур, жодного предмета на тлі")

Навчимо на текстурах другу джерельну модель — ще 25 секунд — і порівняємо
три варіанти на цілі: з нуля, тіло з близького джерела, тіло з далекого.

In [ ]:
torch.manual_seed(0)
far_source = ShapeNet(3)
started = time.perf_counter()
train(far_source, far_x, far_y, epochs=10, learning_rate=1e-3, batch=32, seed=0)
print("навчання далекого джерела: %.1f с, точність на ньому %.3f"
      % (time.perf_counter() - started, accuracy(far_source, far_x, far_y)))
far_state = copy.deepcopy(far_source.state_dict())

In [ ]:
measure_4 = {}
for n in SIZES:
    torch.manual_seed(1)
    far = run(build_transferred(far_state), n, 20, 3e-3)
    # «з нуля» і «близьке джерело» вже пораховані в замірі 1 — тими самими зернами
    measure_4[n] = (measure_1[n][0], measure_1[n][1], far)

print("%10s %10s %16s %14s" % ("прикладів", "з нуля", "близьке джерело", "далеке джерело"))
print("-" * 56)
for n in SIZES:
    a, b, c = measure_4[n]
    print("%10d %10.3f %16.3f %14.3f" % (n, a, b, c))

hurt = [n for n in SIZES if measure_4[n][2] < measure_4[n][0]]
print()
if hurt:
    print("далеке джерело гірше за навчання з нуля на розмірах:", hurt)
    print("це і є ціна невдалого переносу: чуже тіло не помагає, а тримає")
else:
    print("навіть далеке джерело не виявилось гіршим за навчання з нуля")

## 10 · Перші фільтри: чому їх узагалі можна брати чужими

Тепер подивимось на самі ваги. У першого згорткового шару їх вісім штук по 3×3 —
це маленькі шаблони, які мережа прикладає до зображення.

Порівняємо три набори: близьке джерело, далеке джерело й модель, навчену з нуля
на **цілі**. Якщо перші фільтри справді універсальні, набори мають бути схожі.

In [ ]:
torch.manual_seed(1)
target_scratch = ShapeNet(3)
train(target_scratch, target_pool_x, target_pool_y, 20, 3e-3, batch=8, seed=1)

near_filters = source_model.body[0][0].weight.detach().numpy()[:, 0]
far_filters = far_source.body[0][0].weight.detach().numpy()[:, 0]
target_filters = target_scratch.body[0][0].weight.detach().numpy()[:, 0]

figure, axes = plt.subplots(3, 8, figsize=(9, 3.8))
rows = [("джерело: фігури", near_filters),
        ("джерело: текстури", far_filters),
        ("ціль, з нуля", target_filters)]
for row, (title, bank) in enumerate(rows):
    for k in range(8):
        axes[row, k].imshow(bank[k], cmap="RdBu", vmin=-0.6, vmax=0.6)
        axes[row, k].set_xticks([])
        axes[row, k].set_yticks([])
    axes[row, 0].set_ylabel(title, fontsize=8)
plt.tight_layout()
plt.show()
print("три банки по вісім ядер 3×3: червоне — додатні ваги, синє — відʼємні")

«Схожі на око» — слабкий доказ. Порахуємо схожість числом.

Для кожного ядра цільової моделі знайдемо найсхожіше ядро в чужому банку й
візьмемо **косинусну близькість** між ними за модулем. Косинусна близькість — це
міра «дивляться в один бік» для двох наборів чисел: 1 — той самий шаблон,
0 — нічого спільного. Модуль тому, що ядро зі зміненим знаком шукає ту саму
межу, тільки з іншого боку.

In [ ]:
def best_match(bank_a, bank_b):
    """Для кожного ядра з bank_a — найбільша |косинусна близькість| з ядрами bank_b."""
    scores = []
    for i in range(bank_a.shape[0]):
        u = bank_a[i].ravel() - bank_a[i].mean()      # прибираємо середнє: цікавить форма
        best = 0.0
        for j in range(bank_b.shape[0]):
            v = bank_b[j].ravel() - bank_b[j].mean()
            denominator = np.linalg.norm(u) * np.linalg.norm(v)
            if denominator > 1e-9:
                best = max(best, abs(float(u @ v) / denominator))
        scores.append(best)
    return np.array(scores)


random_bank = np.random.default_rng(0).normal(size=near_filters.shape)

print("схожість фільтрів цілі з фільтрами...")
print("  близького джерела : %.3f" % best_match(target_filters, near_filters).mean())
print("  далекого джерела  : %.3f" % best_match(target_filters, far_filters).mean())
print("  випадкового набору: %.3f" % best_match(target_filters, random_bank).mean())
print()
print("випадковий набір — це база порівняння: стільки схожості дає сама лише")
print("розмірність 3×3, без жодного навчання")

Останнє про фільтри — самі числа. Вісім ядер по девʼять ваг, це весь перший шар
мережі. Роздрукуємо їх: далі побачити, що два незалежні навчання прийшли до схожих
табличок, можна вже без картинки.

In [ ]:
def show_bank(title, bank):
    print(title)
    for k in range(bank.shape[0]):
        rows = [" ".join("%5.2f" % v for v in bank[k][r]) for r in range(3)]
        print("  ядро %d: [%s]" % (k, "] [".join(rows)))


show_bank("джерело (фігури):", near_filters)
print()
show_bank("джерело (текстури):", far_filters)
print()
show_bank("ціль, навчена з нуля:", target_filters)

## 11 · Не та нормалізація

Тіло переносять разом із **умовами**, у яких воно навчалось. Найчастіша з них —
нормалізація входу. Наша джерельна модель бачила числа в діапазоні 0..1.
Подамо їй ті самі зображення, оброблені інакше: `(x - 0.5) / 0.5` (діапазон −1..1),
`x * 255` (забули поділити) та `1 - x` (інверсія).

Голова щоразу вчиться заново — тобто в неї є всі шанси пристосуватись.

In [ ]:
NORMALIZATIONS = [
    ("та сама, 0..1", lambda t: t),
    ("(x-0.5)/0.5 → -1..1", lambda t: (t - 0.5) / 0.5),
    ("x*255, забули поділити", lambda t: t * 255.0),
    ("1-x, інверсія", lambda t: 1.0 - t),
]

@torch.no_grad()
def silent_share(model, x):
    """Частка нейронів першого блоку, які на цих даних мовчать (ReLU видала нуль)."""
    after_relu = model.body[0][1](model.body[0][0](x))
    return (after_relu == 0).float().mean().item()


measure_6 = {}
print("%-26s %10s %10s %14s" % ("нормалізація цілі", "перенос", "з нуля", "мовчить нейронів"))
print("-" * 64)
for name, transform in NORMALIZATIONS:
    torch.manual_seed(1)
    transferred = build_transferred(source_state)
    silent = silent_share(transferred, transform(test_x))
    train(transferred, transform(target_pool_x[:24]), target_pool_y[:24], 20, 3e-3,
          batch=8, seed=1, params=trainable(transferred))
    transferred_score = accuracy(transferred, transform(test_x), test_y)

    torch.manual_seed(1)
    scratch_model = build_from_scratch()
    train(scratch_model, transform(target_pool_x[:24]), target_pool_y[:24], 20, 3e-3,
          batch=8, seed=1)
    scratch_score = accuracy(scratch_model, transform(test_x), test_y)

    measure_6[name] = (transferred_score, scratch_score, silent)
    print("%-26s %10.3f %10.3f %13.0f %%"
          % (name, transferred_score, scratch_score, 100 * silent))

print()
print("колонка «з нуля» — контроль: вона показує, що самі по собі ці числа")
print("задачу не ламають. Ламає їх саме розбіжність із тим, що бачило чуже тіло.")

In [ ]:
print("зошит виконався за %.0f с" % (time.perf_counter() - notebook_started))

## Завдання

### 🟢 Рівень 1

Додай до заміру 1 ще дві точки: 6 і 480 цільових прикладів (для 480 поклич
`make_dataset` із більшим `count`). Побудуй графік двох кривих — з нуля й
заморожене тіло — і познач на ньому точку перетину.

**Зроблено, якщо:** на графіку видно, що криві сходяться й перетинаються, і ти
можеш назвати розмір вибірки, після якого перенос перестає бути вигідним.

### 🟡 Рівень 2

Візьми джерело з **шести** класів (усі фігури) замість трьох і повтори замір 1.
Питання: чи зсунеться точка розвороту праворуч, тобто чи стане перенос вигідним
на більших вибірках, якщо джерело було багатше?

**Зроблено, якщо:** у тебе є таблиця на тих самих чотирьох розмірах для джерела
з трьох і з шести класів, і ти пояснив словами, чому вийшло саме так.

### 🔴 Рівень 3

Придумай **третє** джерело — не близьке й не текстурне, а проміжне: наприклад,
ті самі фігури, але сильно зашумлені, або фігури вдвічі меншого розміру.
Перевір, чи схожість джерела й цілі справді монотонно повʼязана з користю переносу.

**Зроблено, якщо:** ти назвав числову міру схожості джерела й цілі (наприклад,
середню косинусну близькість перших фільтрів) і показав на трьох-чотирьох
джерелах, чи корелює вона з приростом точності.